# Arduino Signal Classifier
### Train a neural network in Python, run it on Arduino Uno

**Pipeline:**
1. Generate synthetic signal data
2. Train neural network (PyTorch)
3. Export weights as C arrays
4. Run forward pass on Arduino Uno


## 1. Generate Synthetic Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 16
CLASSES = ['rising', 'falling', 'U', 'inverted_U']
N_SAMPLES = 1000

def generate(cls, n=16, noise=20):
    x = np.linspace(0, 1, n)
    if cls == 'rising':
        s = x
    elif cls == 'falling':
        s = 1 - x
    elif cls == 'U':
        s = 4 * (x - 0.5)**2
        s = (s - s.min()) / (s.max() - s.min())
    elif cls == 'inverted_U':
        s = 1 - 4 * (x - 0.5)**2
        s = (s - s.min()) / (s.max() - s.min())

    offset = np.random.uniform(0, 0.3)
    scale_f = np.random.uniform(0.4, 1.0)
    s = offset + s * scale_f
    s = s * 1023
    s += np.random.randn(n) * noise
    s = np.clip(s, 0, 1023).astype(int)
    return s

# Generate dataset
X_list, y_list = [], []
for idx, cls in enumerate(CLASSES):
    for _ in range(N_SAMPLES):
        X_list.append(generate(cls))
        y_list.append(idx)

X = np.array(X_list)
y = np.array(y_list)

print(f"Dataset: {X.shape}, Classes: {CLASSES}")

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for i, cls in enumerate(CLASSES):
    for _ in range(5):
        axes[i].plot(generate(cls), alpha=0.5)
    axes[i].set_title(cls)
    axes[i].set_ylim(-50, 1100)
    axes[i].grid(True)
plt.tight_layout()
plt.show()

## 2. Train Neural Network

In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test,  dtype=torch.long)

# Model: 16 -> 8 -> 4
model = nn.Sequential(
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 4)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn   = nn.CrossEntropyLoss()

for i in range(500):
    pred = model(X_train)
    loss = loss_fn(pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(f"Step {i:4d}: loss={loss.item():.4f}")

# Evaluate
with torch.no_grad():
    test_pred = model(X_test)
    acc = (test_pred.argmax(1) == y_test).float().mean()
    print(f"\nTest accuracy: {acc*100:.1f}%")

## 3. Export Weights for Arduino

In [ ]:
import os

os.makedirs("sinyal_siniflandirici", exist_ok=True)

W1 = model[0].weight.detach().numpy()
b1 = model[0].bias.detach().numpy()
W2 = model[2].weight.detach().numpy()
b2 = model[2].bias.detach().numpy()

mean  = scaler.mean_.astype('float32')
scale = scaler.scale_.astype('float32')

def arr_to_c(name, arr):
    flat = arr.flatten()
    vals = ', '.join([f'{v:.6f}f' for v in flat])
    return f"float {name}[] = {{{vals}}};\n"

header  = "// Auto-generated by train.ipynb\n\n"
header += "#define N_INPUT  16\n"
header += "#define N_HIDDEN  8\n"
header += "#define N_OUTPUT  4\n\n"
header += arr_to_c("mean", mean)
header += arr_to_c("scale", scale)
header += arr_to_c("W1", W1)
header += arr_to_c("b1", b1)
header += arr_to_c("W2", W2)
header += arr_to_c("b2", b2)

with open("sinyal_siniflandirici/model_weights.h", "w") as f:
    f.write(header)

total = W1.size + b1.size + W2.size + b2.size
print(f"model_weights.h generated!")
print(f"Total floats: {total}")
print(f"Memory: ~{total*4} bytes (Arduino Uno has 2048 bytes RAM)")

## 4. Upload to Arduino

1. Open `sinyal_siniflandirici/sinyal_siniflandirici.ino` in Arduino IDE
2. `model_weights.h` should appear as a tab automatically
3. Select **Arduino Uno** and correct port
4. Upload
5. Open Serial Plotter at **115200 baud**
6. Connect a potentiometer to A0 and turn it!

**Expected output in Serial Plotter:**
- A0 signal trace
- Category indicator jumps to 1000 when signal is classified
